# YOLOv8n Model Training

Train a YOLOv8 nano model for 30 epochs on your violence detection dataset. This notebook loads data from the final folder, organizes it in YOLO format, trains the model, and generates predictions.

## Section 1: Import Required Libraries

In [1]:
import os
import shutil
import cv2
import numpy as np
from pathlib import Path
import random
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
import yaml
import matplotlib.pyplot as plt
from PIL import Image
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print("All libraries imported successfully!")

PyTorch version: 2.12.0
CUDA available: False
All libraries imported successfully!


## Section 2: Load and Prepare Dataset

In [2]:
# Define paths
final_path = Path("/Users/esakkikannan/frames2/final")
frames_path = final_path / "frames"
labels_path = final_path / "labels"
dataset_path = Path("/Users/esakkikannan/frames2/yolo_dataset")

print(f"Final path exists: {final_path.exists()}")
print(f"Frames path exists: {frames_path.exists()}")
print(f"Labels path exists: {labels_path.exists()}")

# Create dataset directory structure
for split in ['train', 'val', 'test']:
    (dataset_path / split / 'images').mkdir(parents=True, exist_ok=True)
    (dataset_path / split / 'labels').mkdir(parents=True, exist_ok=True)

print(f"Dataset directory created at: {dataset_path}")

Final path exists: True
Frames path exists: True
Labels path exists: True
Dataset directory created at: /Users/esakkikannan/frames2/yolo_dataset


In [3]:
# Define class mapping
classes = {
    'assault_violence': 0,
    'gun_violence': 1,
    'normal_actions': 2,
    'sabotage_violence': 3
}

# Get all category folders
category_folders = [d for d in frames_path.iterdir() if d.is_dir() and not d.name.startswith('.')]
category_folders.sort()

print(f"Found {len(category_folders)} category folders")
print("Categories:", [f.name for f in category_folders])

Found 79 category folders
Categories: ['assault_violence (1)', 'assault_violence (10)', 'assault_violence (11)', 'assault_violence (12)', 'assault_violence (13)', 'assault_violence (14)', 'assault_violence (15)', 'assault_violence (16)', 'assault_violence (17)', 'assault_violence (18)', 'assault_violence (19)', 'assault_violence (2)', 'assault_violence (20)', 'assault_violence (3)', 'assault_violence (4)', 'assault_violence (5)', 'assault_violence (6)', 'assault_violence (7)', 'assault_violence (8)', 'assault_violence (9)', 'gun_violence (10)', 'gun_violence (11)', 'gun_violence (12)', 'gun_violence (13)', 'gun_violence (14)', 'gun_violence (15)', 'gun_violence (16)', 'gun_violence (17)', 'gun_violence (18)', 'gun_violence (19)', 'gun_violence (2)', 'gun_violence (20)', 'gun_violence (3)', 'gun_violence (4)', 'gun_violence (5)', 'gun_violence (6)', 'gun_violence (7)', 'gun_violence (8)', 'gun_violence (9)', 'normal_actions (1)', 'normal_actions (10)', 'normal_actions (11)', 'normal_act

In [4]:
# Collect all images and labels
all_images = []
for category_folder in category_folders:
    category_name = category_folder.name
    category_frames = category_folder
    category_labels = labels_path / category_name
    
    if category_labels.exists():
        image_files = sorted([f for f in category_frames.glob('*.jpg')])
        for img_file in image_files:
            label_file = category_labels / (img_file.stem + '.txt')
            if label_file.exists():
                all_images.append({
                    'image': img_file,
                    'label': label_file,
                    'category': category_name,
                    'class_id': classes.get(category_name.split('(')[0].strip(), 0)
                })

print(f"Total image-label pairs found: {len(all_images)}")

# Split data: 70% train, 15% val, 15% test
train_imgs, temp_imgs = train_test_split(all_images, test_size=0.3, random_state=42)
val_imgs, test_imgs = train_test_split(temp_imgs, test_size=0.5, random_state=42)

print(f"Train: {len(train_imgs)}, Val: {len(val_imgs)}, Test: {len(test_imgs)}")

Total image-label pairs found: 22950
Train: 16065, Val: 3442, Test: 3443


In [5]:
def copy_files_to_split(data_list, split_name, dataset_path):
    """Copy images and labels to respective split folders"""
    for item in data_list:
        img_src = item['image']
        label_src = item['label']
        class_id = item['class_id']
        
        # Copy image
        img_dst = dataset_path / split_name / 'images' / img_src.name
        shutil.copy2(img_src, img_dst)
        
        # Read and process label
        with open(label_src, 'r') as f:
            label_content = f.read().strip()
        
        # Create YOLO format label (class_id followed by normalized bbox or other format)
        # Assuming label format is: "class_name x1 y1 x2 y2" or similar
        label_dst = dataset_path / split_name / 'labels' / (img_src.stem + '.txt')
        with open(label_dst, 'w') as f:
            # Write class_id
            f.write(str(class_id))
            # If label has bbox coordinates, append them; otherwise just write class_id
            if ' ' in label_content:
                parts = label_content.split()
                if len(parts) > 1:
                    # Extract coordinates and normalize them
                    try:
                        coords = [float(p) for p in parts[1:]]
                        if len(coords) >= 4:
                            # Normalize coordinates (assuming they're pixel values from 640x480)
                            # Format: x1 y1 x2 y2 -> cx cy w h (normalized)
                            x1, y1, x2, y2 = coords[0], coords[1], coords[2], coords[3]
                            # Get image dimensions
                            img = Image.open(img_src)
                            w, h = img.size
                            # Convert to center coordinates and normalize
                            cx = ((x1 + x2) / 2) / w
                            cy = ((y1 + y2) / 2) / h
                            bbox_w = (x2 - x1) / w
                            bbox_h = (y2 - y1) / h
                            # Clamp to [0, 1]
                            cx = min(max(cx, 0), 1)
                            cy = min(max(cy, 0), 1)
                            bbox_w = min(max(bbox_w, 0), 1)
                            bbox_h = min(max(bbox_h, 0), 1)
                            f.write(f" {cx:.6f} {cy:.6f} {bbox_w:.6f} {bbox_h:.6f}\n")
                        else:
                            f.write("\n")
                    except:
                        f.write("\n")
                else:
                    f.write("\n")
            else:
                f.write("\n")

# Copy files to train, val, test splits
print("Copying training files...")
copy_files_to_split(train_imgs, 'train', dataset_path)
print("Copying validation files...")
copy_files_to_split(val_imgs, 'val', dataset_path)
print("Copying test files...")
copy_files_to_split(test_imgs, 'test', dataset_path)

print("Dataset organization complete!")

Copying training files...
Copying validation files...
Copying test files...
Dataset organization complete!


In [6]:
# Create data.yaml for YOLO
data_yaml = {
    'path': str(dataset_path),
    'train': str(dataset_path / 'train' / 'images'),
    'val': str(dataset_path / 'val' / 'images'),
    'test': str(dataset_path / 'test' / 'images'),
    'nc': len(classes),
    'names': {v: k for k, v in classes.items()}
}

yaml_path = dataset_path / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f"data.yaml created at: {yaml_path}")
print("\nDataset configuration:")
print(yaml.dump(data_yaml))

data.yaml created at: /Users/esakkikannan/frames2/yolo_dataset/data.yaml

Dataset configuration:
names:
  0: assault_violence
  1: gun_violence
  2: normal_actions
  3: sabotage_violence
nc: 4
path: /Users/esakkikannan/frames2/yolo_dataset
test: /Users/esakkikannan/frames2/yolo_dataset/test/images
train: /Users/esakkikannan/frames2/yolo_dataset/train/images
val: /Users/esakkikannan/frames2/yolo_dataset/val/images



## Section 3: Initialize YOLOv8n Model

In [7]:
# Load YOLOv8n model (nano version)
model = YOLO('yolov8n.pt')

print("YOLOv8n model loaded successfully!")
print(f"Model summary: {model.model}")

WARNING ⚠️ Download failure, retrying 1/3 https://github.com/ultralytics/assets/releases/download/v8.4.0/yolov8n.pt... <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1081)>


###################################################################       93.9%

YOLOv8n model loaded successfully!
Model summary: DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C2f(
      (cv1): Conv(
        (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, af

######################################################################## 100.0%


## Section 4: Train Model for 30 Epochs

In [10]:
import torch

device = "mps" if torch.backends.mps.is_available() else "cpu"

results = model.train(
    data=str(yaml_path),
    epochs=30,
    imgsz=640,
    batch=8,
    patience=5,
    save=True,
    device=device,
    verbose=True,
    project='/Users/esakkikannan/frames2/yolo_results',
    name='violence_detection_v8n',
    pretrained=True,
    amp=True
)

print("Training completed!")
print("Results saved to: /Users/esakkikannan/frames2/yolo_results")

New https://pypi.org/project/ultralytics/8.4.58 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.56 🚀 Python-3.14.4 torch-2.12.0 MPS (Apple M5)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/esakkikannan/frames2/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=vi

## Section 5: Evaluate Model Performance

In [16]:
# Evaluate model on validation set
metrics = model.val()

print("="*60)
print("VALIDATION METRICS - MODEL ACCURACY")
print("="*60)

# Overall metrics
print(f"\nOverall mAP Scores:")
print(f"  mAP@0.5 (IoU=0.5): {metrics.box.map50:.4f}")
print(f"  mAP@0.5:0.95 (IoU=0.5:0.95): {metrics.box.map:.4f}")
print(f"  Fitness Score: {metrics.fitness:.4f}")

# Per-class metrics
print(f"\nPer-Class Metrics:")
class_names = list(classes.values())
class_ids = list(classes.keys())

for i, class_name in enumerate(class_ids):
    print(f"\n  {class_name}:")
    print(f"    Precision: {metrics.box.p[i]:.4f}")
    print(f"    Recall: {metrics.box.r[i]:.4f}")
    print(f"    F1-Score: {metrics.box.f1[i]:.4f}")
    print(f"    AP@0.5: {metrics.box.ap50[i]:.4f}")
    print(f"    AP@0.5:0.95: {metrics.box.ap[i]:.4f}")

# Overall averages
print(f"\nOverall Averages:")
print(f"  Mean Precision: {metrics.box.p.mean():.4f}")
print(f"  Mean Recall: {metrics.box.r.mean():.4f}")
print(f"  Mean F1-Score: {metrics.box.f1.mean():.4f}")

print("\n" + "="*60)

Ultralytics 8.4.56 🚀 Python-3.14.4 torch-2.12.0 CPU (Apple M5)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 422.6±170.3 MB/s, size: 84.9 KB)
val: Scanning /Users/esakkikannan/frames2/yolo_dataset/val/labels.cache... 297 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 297/297 49.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 1.3s/it 24.7s1.4ss
                   all        297        297      0.989      0.981      0.994      0.994
      assault_violence         74         74       0.96          1      0.993      0.993
          gun_violence         77         77          1      0.931      0.994      0.994
        normal_actions         59         59          1      0.992      0.995      0.995
     sabotage_violence         87         87      0.997          1      0.995      0.995
Speed: 0.2ms preprocess, 80.2ms inference, 0.0ms loss, 0.5ms postprocess per image
Results saved to /Users/esakkikannan/fr

## Section 6: Make Predictions with Final Model

In [17]:
# Load the best trained model
best_model_path = Path('/Users/esakkikannan/frames2/yolo_results/violence_detection_v8n/weights/best.pt')
if best_model_path.exists():
    model = YOLO(str(best_model_path))
    print(f"Best model loaded from: {best_model_path}")
else:
    print("Best model not found, using the trained model")

# Make predictions on test images
test_images_path = dataset_path / 'test' / 'images'
test_image_files = list(test_images_path.glob('*.jpg'))[:10]  # Test on first 10 images

print(f"\nMaking predictions on {len(test_image_files)} test images...")

for img_path in test_image_files:
    results = model.predict(source=str(img_path), conf=0.25)
    print(f"\nPrediction for {img_path.name}:")
    for result in results:
        print(f"  Boxes: {result.boxes}")
        print(f"  Confidences: {result.boxes.conf}")
        if hasattr(result, 'names'):
            for box, conf in zip(result.boxes.xyxy, result.boxes.conf):
                print(f"    {result.names[int(result.boxes.cls[0])]} - Confidence: {conf:.2f}")

Best model not found, using the trained model

Making predictions on 10 test images...

image 1/1 /Users/esakkikannan/frames2/yolo_dataset/test/images/frame_215.jpg: 640x480 1 gun_violence, 47.0ms
Speed: 1.2ms preprocess, 47.0ms inference, 0.4ms postprocess per image at shape (1, 3, 640, 480)

Prediction for frame_215.jpg:
  Boxes: ultralytics.engine.results.Boxes object with attributes:

cls: tensor([1.])
conf: tensor([0.9817])
data: tensor([[  3.5553,   0.0000, 477.6805, 640.0000,   0.9817,   1.0000]])
id: None
is_track: False
orig_shape: (640, 480)
shape: torch.Size([1, 6])
xywh: tensor([[240.6179, 320.0000, 474.1252, 640.0000]])
xywhn: tensor([[0.5013, 0.5000, 0.9878, 1.0000]])
xyxy: tensor([[  3.5553,   0.0000, 477.6805, 640.0000]])
xyxyn: tensor([[0.0074, 0.0000, 0.9952, 1.0000]])
  Confidences: tensor([0.9817])
    gun_violence - Confidence: 0.98

image 1/1 /Users/esakkikannan/frames2/yolo_dataset/test/images/frame_201.jpg: 640x480 1 gun_violence, 19.0ms
Speed: 0.4ms preprocess,

In [ ]:
# Visualize predictions with bounding boxes (filtered by class)
fig, axes = plt.subplots(2, 5, figsize=(20, 10))
axes = axes.flatten()

# Define which classes to visualize (change this to filter)
classes_to_show = list(classes.keys())  # Show all classes

for idx, img_path in enumerate(test_image_files[:10]):
    results = model.predict(source=str(img_path), conf=0.25)
    
    # Load original image
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Draw predictions (filtered by class)
    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = box.conf[0]
            cls_id = int(box.cls[0])
            
            # Get class name
            class_name = result.names[cls_id] if hasattr(result, 'names') else f'Class {cls_id}'
            
            # Only draw if in our filter list
            if class_name in classes_to_show:
                # Draw bounding box
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                
                # Put label
                label = f'{class_name}: {conf:.2f}'
                cv2.putText(img, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    
    # Display image
    axes[idx].imshow(img)
    axes[idx].set_title(img_path.name)
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('/Users/esakkikannan/frames2/predictions_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

print("Predictions visualization saved!")
print(f"Showing classes: {classes_to_show}")


image 1/1 /Users/esakkikannan/frames2/yolo_dataset/test/images/frame_215.jpg: 640x480 1 gun_violence, 22.9ms
Speed: 0.7ms preprocess, 22.9ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 480)

image 1/1 /Users/esakkikannan/frames2/yolo_dataset/test/images/frame_201.jpg: 640x480 1 gun_violence, 19.6ms
Speed: 0.4ms preprocess, 19.6ms inference, 0.3ms postprocess per image at shape (1, 3, 640, 480)

image 1/1 /Users/esakkikannan/frames2/yolo_dataset/test/images/frame_229.jpg: 640x480 1 sabotage_violence, 20.0ms
Speed: 0.4ms preprocess, 20.0ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 480)

image 1/1 /Users/esakkikannan/frames2/yolo_dataset/test/images/frame_59.jpg: 640x480 1 gun_violence, 18.6ms
Speed: 0.4ms preprocess, 18.6ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 480)

image 1/1 /Users/esakkikannan/frames2/yolo_dataset/test/images/frame_71.jpg: 640x480 1 normal_actions, 19.1ms
Speed: 0.4ms preprocess, 19.1ms inference, 0.2ms postproc

<Figure size 2000x1000 with 10 Axes>

Predictions visualization saved!
All training and prediction tasks completed successfully!


In [19]:
# Plot model accuracy metrics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

class_names_list = list(classes.keys())

# 1. Precision per class
ax = axes[0, 0]
ax.bar(class_names_list, metrics.box.p)
ax.set_ylabel('Precision')
ax.set_title('Precision by Class')
ax.set_ylim([0, 1])
for i, v in enumerate(metrics.box.p):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

# 2. Recall per class
ax = axes[0, 1]
ax.bar(class_names_list, metrics.box.r, color='orange')
ax.set_ylabel('Recall')
ax.set_title('Recall by Class')
ax.set_ylim([0, 1])
for i, v in enumerate(metrics.box.r):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

# 3. F1-Score per class
ax = axes[1, 0]
ax.bar(class_names_list, metrics.box.f1, color='green')
ax.set_ylabel('F1-Score')
ax.set_title('F1-Score by Class')
ax.set_ylim([0, 1])
for i, v in enumerate(metrics.box.f1):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

# 4. AP scores
ax = axes[1, 1]
x = np.arange(len(class_names_list))
width = 0.35
ax.bar(x - width/2, metrics.box.ap50, width, label='AP@0.5', color='skyblue')
ax.bar(x + width/2, metrics.box.ap, width, label='AP@0.5:0.95', color='lightcoral')
ax.set_ylabel('AP Score')
ax.set_title('Average Precision by Class')
ax.set_xticks(x)
ax.set_xticklabels(class_names_list, rotation=45, ha='right')
ax.set_ylim([0, 1])
ax.legend()

plt.tight_layout()
plt.savefig('/Users/esakkikannan/frames2/accuracy_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

print("Accuracy metrics visualization saved!")

<Figure size 1400x1000 with 4 Axes>

Accuracy metrics visualization saved!


## Section 7: Test with Video

In [27]:
# Find video files in the workspace
from pathlib import Path
import glob

# Search for video files
video_extensions = ['*.mp4', '*.avi', '*.mov', '*.mkv', '*.flv']
video_files = []

# Search in common locations
search_paths = [
    '/Users/esakkikannan/frames2',
    '/Users/esakkikannan/frames2/frames generated',
    '/Users/esakkikannan'
]

for search_path in search_paths:
    for ext in video_extensions:
        found = glob.glob(f"{search_path}/{ext}")
        video_files.extend(found)
    # Also search recursively one level deep
    for ext in video_extensions:
        found = glob.glob(f"{search_path}/*/{ext}")
        video_files.extend(found)

# Remove duplicates
video_files = list(set(video_files))

if video_files:
    print(f"Found {len(video_files)} video file(s):")
    for i, video in enumerate(video_files):
        print(f"  {i}: {video}")
    
    # Use the first video found
    video_path = video_files[0]
    print(f"\nUsing video: {video_path}")
else:
    print("No video files found. Specify your video path manually:")
    # You can set this manually if needed
    video_path = None
    print("To use a video, set video_path = '/path/to/your/video.mp4'")

Found 3 video file(s):
  0: /Users/esakkikannan/frames2/assault_violence (2).mp4
  1: /Users/esakkikannan/frames2/assault_violence (2)_predictions.mp4
  2: /Users/esakkikannan/Downloads/RAW #shooting caught on Lemay security #camera - KSDK News (720p, h264).mp4

Using video: /Users/esakkikannan/frames2/assault_violence (2).mp4


In [ ]:
# Process video with YOLO model (with class filtering)
# Note: Configure classes_to_detect in the cell above before running this

print(f"Processing video: {video_path}")
print(f"Filtering to show only: {classes_to_detect}\n")

if video_path and Path(video_path).exists():
    cap = cv2.VideoCapture(video_path)
    
    # Get video properties
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    print(f"Video Properties:")
    print(f"  FPS: {fps}")
    print(f"  Resolution: {width}x{height}")
    print(f"  Total Frames: {frame_count}")
    print(f"  Duration: {frame_count/fps:.2f} seconds")
    
    # Output video path
    output_video_path = Path(video_path).stem + '_predictions.mp4'
    output_video_path = str(Path('/Users/esakkikannan/frames2') / output_video_path)
    
    # Define the codec and create VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))
    
    frame_num = 0
    predictions_summary = {cls: 0 for cls in classes_to_detect}
    
    print(f"\nProcessing frames...")
    
    while cap.isOpened():
        ret, frame = cap.read()
        
        if not ret:
            break
        
        # Run YOLO prediction on frame
        results = model.predict(source=frame, conf=0.25, verbose=False)
        
        # Draw predictions on frame (only for selected classes)
        for result in results:
            for box in result.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                conf = box.conf[0]
                cls_id = int(box.cls[0])
                
                # Get class name
                class_name = result.names[cls_id] if hasattr(result, 'names') else f'Class {cls_id}'
                
                # Only draw if this class is in our filter list
                if class_name in classes_to_detect:
                    # Draw bounding box
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    
                    # Put label with confidence
                    label = f'{class_name}: {conf:.2f}'
                    cv2.putText(frame, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
                    
                    # Count predictions
                    if class_name in predictions_summary:
                        predictions_summary[class_name] += 1
        
        # Write frame to output video
        out.write(frame)
        
        frame_num += 1
        if frame_num % 30 == 0:
            print(f"  Processed {frame_num}/{frame_count} frames...")
    
    # Release everything
    cap.release()
    out.release()
    
    print(f"\nVideo processing completed!")
    print(f"Output video saved to: {output_video_path}")
    print(f"\nDetections Summary (filtered):")
    for class_name, count in predictions_summary.items():
        print(f"  {class_name}: {count} detections")
    
else:
    print("Video file not found. Please specify a valid video path.")

Processing video: /Users/esakkikannan/frames2/assault_violence (2).mp4
Video Properties:
  FPS: 30
  Resolution: 480x640
  Total Frames: 296
  Duration: 9.87 seconds

Processing frames...
  Processed 30/296 frames...
  Processed 60/296 frames...
  Processed 90/296 frames...
  Processed 120/296 frames...
  Processed 150/296 frames...
  Processed 180/296 frames...
  Processed 210/296 frames...
  Processed 240/296 frames...
  Processed 270/296 frames...

Video processing completed!
Output video saved to: /Users/esakkikannan/frames2/assault_violence (2)_predictions.mp4

Detections Summary:
  assault_violence: 296 detections
  gun_violence: 0 detections
  normal_actions: 0 detections
  sabotage_violence: 0 detections


In [30]:
# Specify video path and detection classes
# Paste your video path here:

video_path = '/Users/esakkikannan/frames2/RAW #shooting caught on Lemay security #camera - KSDK News (720p, h264).mp4'

# Choose which classes to detect (show bounding boxes for)
# Options: 'assault_violence', 'gun_violence', 'normal_actions', 'sabotage_violence'
classes_to_detect = ['assault_violence', 'gun_violence']  # Only show these classes

print(f"Video path: {video_path}")
print(f"File exists: {Path(video_path).exists() if isinstance(video_path, str) else 'Webcam mode'}")
print(f"\nClasses to detect (draw boxes for): {classes_to_detect}")
print("Classes will be filtered - only selected ones will show bounding boxes")

Video path: /Users/esakkikannan/frames2/RAW #shooting caught on Lemay security #camera - KSDK News (720p, h264).mp4
File exists: True

Classes to detect (draw boxes for): ['assault_violence', 'gun_violence']
Classes will be filtered - only selected ones will show bounding boxes


In [31]:
# Run video detection with class filtering
print(f"Starting video detection on: {video_path}")
print(f"Filtering for classes: {classes_to_detect}\n")

if isinstance(video_path, str) and Path(video_path).exists():
    cap = cv2.VideoCapture(video_path)
    
    # Get video properties
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    print(f"Video Details:")
    print(f"  Resolution: {width}x{height}")
    print(f"  FPS: {fps}")
    print(f"  Total Frames: {frame_count}")
    print(f"  Duration: {frame_count/fps:.2f} seconds\n")
    
    # Output video path
    video_name = Path(video_path).stem
    output_video_path = f'/Users/esakkikannan/frames2/{video_name}_detected.mp4'
    
    # Create video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))
    
    frame_num = 0
    predictions_summary = {cls: 0 for cls in classes_to_detect}
    
    print("Processing frames...")
    
    while cap.isOpened():
        ret, frame = cap.read()
        
        if not ret:
            break
        
        # Run YOLO detection
        results = model.predict(source=frame, conf=0.25, verbose=False)
        
        # Draw detections (only for selected classes)
        for result in results:
            for box in result.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                conf = box.conf[0]
                cls_id = int(box.cls[0])
                
                # Get class name
                class_name = result.names[cls_id] if hasattr(result, 'names') else f'Class {cls_id}'
                
                # Only draw if this class is in our filter list
                if class_name in classes_to_detect:
                    # Draw bounding box
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    
                    # Put label with confidence
                    label = f'{class_name}: {conf:.2f}'
                    cv2.putText(frame, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
                    
                    # Count detections
                    if class_name in predictions_summary:
                        predictions_summary[class_name] += 1
        
        # Write frame
        out.write(frame)
        
        frame_num += 1
        if frame_num % 30 == 0:
            print(f"  Processed {frame_num}/{frame_count} frames...")
    
    cap.release()
    out.release()
    
    print(f"\n✓ Detection completed!")
    print(f"Output saved to: {output_video_path}")
    print(f"\nDetection Summary (filtered classes only):")
    print(f"  Total frames processed: {frame_num}")
    for class_name, count in predictions_summary.items():
        print(f"  {class_name}: {count} detections")
    
elif isinstance(video_path, int):
    print("Webcam mode detected. Opening webcam...")
    # Webcam processing would go here
    
else:
    print(f"Error: Video file not found at: {video_path}")
    print("Please check the path and try again.")

Starting video detection on: /Users/esakkikannan/frames2/RAW #shooting caught on Lemay security #camera - KSDK News (720p, h264).mp4
Filtering for classes: ['assault_violence', 'gun_violence']

Video Details:
  Resolution: 720x1280
  FPS: 29
  Total Frames: 598
  Duration: 20.62 seconds

Processing frames...
  Processed 30/598 frames...
  Processed 60/598 frames...
  Processed 90/598 frames...
  Processed 120/598 frames...
  Processed 150/598 frames...
  Processed 180/598 frames...
  Processed 210/598 frames...
  Processed 240/598 frames...
  Processed 270/598 frames...
  Processed 300/598 frames...
  Processed 330/598 frames...
  Processed 360/598 frames...
  Processed 390/598 frames...
  Processed 420/598 frames...
  Processed 450/598 frames...
  Processed 480/598 frames...
  Processed 510/598 frames...
  Processed 540/598 frames...
  Processed 570/598 frames...

✓ Detection completed!
Output saved to: /Users/esakkikannan/frames2/RAW #shooting caught on Lemay security #camera - KSDK 